*v5 - filtered to modeling population, added download_file function, added pip installs*

# Downloading and Parsing Corrections Letters

* This code is split into 5 parts:
* Part A - defining functions
* Part B - load and filter building permits to modeling population (~14k permits)
* Part C - For each permit, find its correction letter URLs (the PDFs)
* Part D - Download the correction letter PDFs
* Part E - Parse the saved corrections letters into correction_comments.csv

Each part is designed so it can run independently, assuming the relevant files/variables are set.

**Run in stages:** flip one flag to True at a time — do_lookup first, then do_download, then do_parsing.

In [1]:
import sys
!{sys.executable} -m pip install pypdf python-dotenv requests --quiet

import os
import re
from pathlib import Path

import pandas as pd
from pandas import json_normalize

import requests
import urllib.parse
from urllib.request import Request, urlopen
from urllib.parse import quote_plus

from pypdf import PdfReader

from dataclasses import dataclass, field, InitVar
from dotenv import load_dotenv

print('Libraries loaded.')


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\flori\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Libraries loaded.


### Part A2 - Control block

In [2]:
# Set static variables
base_directory    = r'C:\Users\flori\Documents\GitHub\CSB425-City-of-Seattle-Permit-Predictor\output'
subdirectory      = ''
working_directory = os.path.join(base_directory, subdirectory)

# Path to master dataset (modeling population)
MASTER_CSV = r'C:\Users\flori\Documents\GitHub\CSB425-City-of-Seattle-Permit-Predictor\data\master_dataset.csv'

# Key file names — saved in output/
correction_letters_file = os.path.join(working_directory, 'correction_letters.csv')
correction_comments     = os.path.join(working_directory, 'correction_comments.csv')

# ── Control flags — flip one at a time ──
do_lookup   = False   # Part C: look up correction letter URLs for each permit
do_download = False   # Part D: download the PDFs
do_parsing  = False   # Part E: parse PDFs into correction_comments.csv
do_debug    = False   # verbose output

print('Control block set.')
print(f'Working directory : {working_directory}')
print(f'do_lookup         : {do_lookup}')
print(f'do_download       : {do_download}')
print(f'do_parsing        : {do_parsing}')


Control block set.
Working directory : C:\Users\flori\Documents\GitHub\CSB425-City-of-Seattle-Permit-Predictor\output\
do_lookup         : False
do_download       : False
do_parsing        : False


In [3]:
print(f"The files in the working directory are {os.listdir(working_directory)}")

The files in the working directory are ['comment_coverage_distribution.png', 'comment_features.csv', 'cv_metrics_per_fold.png', 'cv_metrics_per_fold_comments.png', 'cv_metrics_per_fold_xgb.png', 'delay_drivers.png', 'DiagnosticReport_CommentFeatures.txt', 'DiagnosticReport_FeatureSelection.txt', 'DiagnosticReport_RF.txt', 'DiagnosticReport_RF_Comments.txt', 'DiagnosticReport_XGB_Comments.txt', 'feature_importance.png', 'feature_importance_comments.png', 'feature_importance_xgb.png', 'feature_selection_results.csv', 'feature_selection_shap_bar.png', 'feature_selection_shap_summary.png', 'feature_selection_stage1.png', 'matched_vs_unmatched.png', 'ModelWeights_RF.joblib', 'ModelWeights_RF_Comments.joblib', 'ModelWeights_RF_SF.joblib', 'ModelWeights_XGB_Comments.joblib', 'oof_pred_vs_actual.png', 'oof_pred_vs_actual_comments.png', 'oof_pred_vs_actual_xgb.png', 'permit_complexity.png', 'permit_map.html', 'residual_analysis.png', 'residual_analysis_comments.png', 'residual_analysis_xgb.png'

### Part A3 - Defining functions

**First, utility functions**

In [4]:
def save_data(df, dest_file, header=True, index=False):
    print(f"Saving data to {dest_file}")
    df.to_csv(dest_file, header=header, index=index)

**Second, download functions**

In [5]:
#2/ function to download documents
def get_correction_letter_url(RecordNumber):
    
    url = "https://web.seattle.gov/ecm/SearchService/searchByRef_Public/{record}".format(record = RecordNumber)
    
    response = requests.get(url) 
    data = response.json() 
    df = pd.DataFrame(data)
    df = df[df['documentType'] == 'Correction Letter']
    #df = df[df['documentTitle'].str.contains('Drainage', regex=True)]

    df['RecordNumber'] = RecordNumber
    df['URL'] = "https://web.seattle.gov/dpd/edms/GetDocument?id=" + df['id'].astype(str)

    return df

In [6]:
def download_file(url, file_path):
    """Download a file from a URL and save to disk."""
    response = requests.get(url, stream=True)
    response.raise_for_status()
    with open(file_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)


review_file = 'Building_Permits_20260416.csv'

if not Path(review_file).is_file():
    print(f'File {review_file} does not exist — check that it is in the working directory: {working_directory}')
else:
    print(f'Found: {review_file}')

In [7]:
# Raw building permits file — lives in data/, not output/
review_file = r'C:\Users\flori\Documents\GitHub\CSB425-City-of-Seattle-Permit-Predictor\data\Building_Permits_20260416.csv'

if not Path(review_file).is_file():
    print(f'File not found: {review_file}')
    print('Download Building_Permits_20260416.csv from the Seattle Open Data portal and place it in the data/ folder.')
else:
    print(f'Found: {review_file}')


File not found: C:\Users\flori\Documents\GitHub\CSB425-City-of-Seattle-Permit-Predictor\data\Building_Permits_20260416.csv
Download Building_Permits_20260416.csv from the Seattle Open Data portal and place it in the data/ folder.


In [8]:
if do_lookup:
    # Load full building permits file
    df_permits = pd.read_csv(review_file)
    df_permits = df_permits.drop_duplicates(subset='RecordNumber')
    print(f'Full building permits file: {len(df_permits):,} rows')

    # ── Filter to modeling population only ──
    # Load permit numbers from master_dataset.csv (~14k completed permits)
    df_master = pd.read_csv(MASTER_CSV, usecols=['permitnum'])
    modeling_permits = set(df_master['permitnum'].str.strip().str.upper())
    print(f'Modeling population size: {len(modeling_permits):,} permits')

    # Normalize RecordNumber to match permitnum format
    df_permits['RecordNumber'] = df_permits['RecordNumber'].str.strip().str.upper()
    df_permits = df_permits[df_permits['RecordNumber'].isin(modeling_permits)]
    print(f'After filter: {len(df_permits):,} permits to process')


## Part C - Downloading files

For each building permit, find and download its correction letter *locations*

In [9]:
#3/ Create list of correction letters and their locations - Load permit list
# get the permit list and de-dup it
if do_lookup:  
    # re-runnable
    if Path(correction_letters_file).is_file():
        df_correction_letters = pd.read_csv(correction_letters_file)
        records_processed = df_correction_letters['RecordNumber'].unique()
        print(f"Previously processed {len(records_processed)} records")
    else:
        df_correction_letters = pd.DataFrame()
        records_processed = []
    
    # get the corrections letters
    num_processed = 0
    for index, row in df_permits.iterrows():
        # check to see if we have it loaded already:
        if not row['RecordNumber'] in records_processed:
            df = get_correction_letter_url(row['RecordNumber'])
            df_correction_letters = pd.concat([df_correction_letters, df])
            num_processed += 1
            if (num_processed % 20 == 0):
                print(f"Processed {num_processed} records")
    
    # format the results
    df_correction_letters['documentDate'] = pd.to_datetime(df_correction_letters['documentDate'])
    df_correction_letters = df_correction_letters[df_correction_letters['documentDate'].dt.year > 2019]
    
    # save the results
    df_correction_letters.to_csv(correction_letters_file, mode='w', header=True, index=True)
else:
    print(f"Skipping correction letter lookups b/c do_download={do_download}")

Skipping correction letter lookups b/c do_download=False


if do_lookup:
    df_correction_letters.to_csv(correction_letters_file, mode='w', header=True, index=True)

## Part D - Download the Correction Letter PDFs

In [10]:
#4/ Download PDFs to the document folder

# TODO - THERE DUPLICATES IN HERE
if do_download:
    df_correction_letters_src = pd.read_csv(correction_letters_file)
    df_correction_letters = df_correction_letters_src.drop_duplicates()

    print(f"There are {len(df_correction_letters)} URLs")
    print (df_correction_letters.columns)
    #df_correction_letters['documentDate'] = pd.to_datetime(df_correction_letters['documentDate'])
    #df_correction_letters = df_correction_letters[df_correction_letters['documentDate'].dt.year > 2024]
    
    df_comments= pd.DataFrame()
    n_downloaded = 0
    n_already = 0
    for index, row in df_correction_letters.iterrows():
        if ((n_downloaded + n_already) % 100 == 0):
            print(f"Downloaded {n_downloaded + n_already}: {n_downloaded} & already-have {n_already} PDFs")
        try:
            original_name = row['originalName']
            record_number = row['RecordNumber']
            url = row['URL']
            file_path = os.path.join(working_directory, original_name)

            #if n_downloaded % 100 == 0 && n_downloaded > 0:
            #    print(f"Downloaded {n_downloaded} files")
            #if n_already % 100 == 0 && n_already > 0:
            #    print(f"Already had {n_already} files")
            
            # skip files we already have
            if Path(file_path).is_file():
                if do_debug:
                    print(f"{record_number} exists, skipping download")
                n_already += 1
            else:
                if do_debug:
                    print(f"{index} Next: {record_number} | url {url}, file {file_path}")
                download_file(url,
                              file_path)
                if do_debug: 
                    print(f"{index} Downloaded")
                n_downloaded += 1
        except Exception as e:
            print (f"Unable to download, Exception {e}")
else:
    print(f"Skipping correction letter downloads b/c do_download={do_download}")

Skipping correction letter downloads b/c do_download=False


## Part E - parsing the saved corrections letters

In [11]:
# function to get comments from pdfs
def get_comments(recordNumber, originalName, documentTitle, url, file_path):

    ReviewTypes = ['Addressing', 'City Light', 'Conveyance', 'Drainage', 
               'ECA GeoTech', 'ECA Riparian', 'ECA Wetland', 'ECA Wildlife', 
               'Energy', 'Finance/Admin Services ADA', 'Fire', 'Floodplain', 
               'Geo Soils', 'Housing', 'Incentive Zoning', 'Land Use', 'Law', 'Mechanical', 'MHA',
               'Neighborhoods', 'Noise', 'Ordinance Structural', 'Ordinance/Structural', 'Ordinance', 
               'Parks', 'Policy', 'Revegetation', 'Shoreline', 
               'Shoring - Private Property', 'Shoring - Right of Way', 
               'Side Sewer Conflict', 'Structural Engineer', 'Sustainability', 
               'Transportation', 'Transportation Management', 'Tree', 'Zoning',
               'Arborist']
    
    RecordNumber = []
    DocumentTitle = []
    Subject = []
    Author = []
    ReviewType = []
    Comment = []
    Cycle = []
    URL = []
    
    # Regex Cycle Number
    #match = re.search(r'\bcycle[\s\-#]*(\d+)', documentTitle, re.IGNORECASE)
    match = re.search(r'cycle[\s_\-#]*(\d+)', documentTitle, re.IGNORECASE)

    cycle = int(match.group(1)) if match else None

    #for file in onlyfiles:
        #print (file)
        # creating a pdf reader object 
    reader = PdfReader(file_path) 

    # creating a page object
    for page in reader.pages:
        text = page.extract_text()

        #print (text)
        comments_list = text.split("Subject: ")
        comments_list.pop(0)

        for i, item in enumerate(comments_list):
            item = "Subject: " + item

            if "Review Type" in item: 

                # clean up variations
                item = re.sub(r'\s*[XY]:\s*[\d.]+\s*in\n?', '', item)
                item = re.sub(r'(?<![\n\s])(Review Type:)', r'\n\1', item)
                item = re.sub(r'(?<![\n\s])(Author:)', r'\n\1', item)
                item = re.sub(r'(?<![\n\s])(Subject:)', r'\n\1', item)
                item = re.sub(r'(?<![\n\s])(Layer:)', r'\n\1', item)
                item = re.sub(r'(?<![\n\s])(X:)', r'\n\1', item)
                item = re.sub(r'(?<![\n\s])(Y:)', r'\n\1', item)


                ## TO ADD: page index. Sheet. 
                # include the code reference as wel. 
                # subject = item.split("Page Index: ")[0]
                # author = item.split("Author: ")[1].split("\n")[0]
                
                subject = re.search(r'Subject:\s*(.*)', item).group(1)
                author = re.search(r'Author:\s*(.*)', item).group(1)
                subject = subject.replace("\n", "")
                author = author.replace("\n", "")
                
                # Extract the comment, account for different formats
                #print(item)
                #print(url)
                
                if item.find("Review Type: ") < item.find("Page Index: "): # Review Type comes first
                    
                    number = re.search(r'\bPage Index:\s*(\d+)', item, re.IGNORECASE)
                    number = int(number.group(1)) if match else None
                    page_index = f"Page Index: {number}"
                    comment = item.split(page_index)[1]
                                        
                    reviewType = re.search(r'Review Type:\s*(.*)', item).group(1)

                else:
                    comment = item.split("Review Type: ")[1] # Review Type comes last

                    # Get review type from list
                    reviewType = next((p for p in ReviewTypes if comment.lower().startswith(p.lower())), None)
                    
                    # Remove the match from the start of the comment (if found)
                    if reviewType:
                        comment = comment[len(reviewType):].lstrip()
                    else:
                        comment = comment
                        

                comment = comment.replace("\n\n", "&&&")
                comment = comment.replace("\n", " ")
                comment = comment.replace("&&&", "\n\n")

                #print (i + 1, subject, comment)

                RecordNumber.append(recordNumber)
                DocumentTitle.append(documentTitle)
                Subject.append(subject)
                Author.append(author)
                Comment.append(comment)
                ReviewType.append(reviewType)
                Cycle.append(cycle)
                URL.append(url)

    # save to dataframe and return
    record = {'RecordNumber': RecordNumber, 'DocumentTitle': DocumentTitle, 'URL': URL, 'Subject':Subject,
            'Author': Author, 'ReviewType': ReviewType, 'Cycle':Cycle, 'Comment':Comment} 
    df = pd.DataFrame(record)
    
    return df

In [12]:
# TEST/DEBUG THE ISSUE
# FIRST STEP, REPRODUCE THE ISSUE

In [ ]:
# Get comments from list of correction letters
if not Path(correction_letters_file).is_file():
    print(f'correction_letters.csv not found at: {correction_letters_file}')
    print('Run Part C first (set do_lookup = True) to generate it.')
else:
    df_correction_letters = pd.read_csv(correction_letters_file)

    # load in the comments file if it exists, otherwise make a new dataframe
    if Path(correction_comments).is_file():
        df_comments = pd.read_csv(correction_comments)
    else:
        df_comments = pd.DataFrame()

    total_docs = len(df_correction_letters['originalName'].drop_duplicates())
    docs_processed = 0

    if do_parsing:
        for index, row in df_correction_letters.iterrows():
            if docs_processed % 100 == 0:
                print(f"Processed {docs_processed} documents, {round(docs_processed * 100 / total_docs,1)} %")
                df_comments.to_csv(correction_comments, mode='w', header=True, index=False)
            try:
                df = get_comments(row['RecordNumber'],
                                  row['originalName'],
                                  row['documentTitle'],
                                  row['URL'],
                                  os.path.join(working_directory, row['originalName']))
                df_comments = pd.concat([df_comments, df])
                docs_processed += 1
            except Exception as e:
                print(f"Unable to parse, Exception {e}")

        df_comments = df_comments.drop_duplicates(subset=df_comments.columns.difference(['URL']))
        df_comments.to_csv(correction_comments, mode='w', header=True, index=False)
        print(f'Done. Saved {len(df_comments):,} comment rows to {correction_comments}')
    else:
        print(f'Skipping parsing (do_parsing={do_parsing}). {len(df_correction_letters):,} correction letters loaded.')

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\flori\\Documents\\GitHub\\CSB425-City-of-Seattle-Permit-Predictor\\output\\correction_letters.csv'